# Suite EXP — Expressions

An expression is an `Expressions.Framework.Expression`, a reference object whose `root` is a term: a value of
`Expressions.Framework.Term`, a union of the kinds `literal`, `call`, `variable` and `let`, to which Basic adds
`quantifier` by schema extension. `Framework.install` and `Basic.install` define them in a store; terms are built there,
in a transaction, through mbse-schemas' builders (or Basic's `Writer`s, which write through them), and read through its
readers, so they serialize, validate and compare like any other value. `Evaluators` computes them (suite EVL).

In [ ]:
import math

from mbse.Expressions import Framework
from mbse.Expressions.Dialects import Basic
from mbse.Expressions.Dialects.Basic import Evaluators as V, Expressions as E
from mbse.Expressions.Framework import Terms as F
from mbse.Schemas.Framework import Builders, Comparison as C, JSON, Plain, Validators, YAML
from support import raises

store = Builders.Store()
store.transaction(Framework.install)
store.transaction(Basic.install)
Term = store.registered("Expressions.Framework.Term")
Expression = store.registered("Expressions.Framework.Expression")


def term(spec):
    """The term value `spec` writes, built in a transaction."""
    with store.transaction():
        return store.Expressions.Framework.Term(E.DIALECT.spec(spec))


lit = term(E.literal(3))
op = term(E.operation("f", 1, "x", lit))

## EXP-01 · Literals hold one basic scalar

In [ ]:
assert F.kind_of(lit)[0] == "literal" and F.form(lit) == F.Form("literal", {"value": 3}) and E.DIALECT.validate(lit) == []
assert lit.literal.value.get() == 3 and lit.as_schema() is Term  # read through the readers
for value in [0, 2**70, -0.0, math.nan, "", b"", False]:  # every basic scalar
    assert E.DIALECT.validate(term(value)) == [] and F.same(term(value), term(E.literal(value)))
assert type(F.form(term(-0.0)).attributes["value"]) is float and math.copysign(1.0, term(-0.0).literal.value.get()) == -1.0
assert E.DIALECT.validate(term(E.literal(None))) == ["a literal needs a value"]
assert E.DIALECT.validate(term(lambda u: u.literal(lambda r: r))) == ["a literal needs a value"]
assert F.same(term(lambda u: u.literal(lambda r: r.value(lambda p: p.as_scalar("x")))), term("x"))  # the builders' own calls
# value is an OfAny of the five basic scalars: its schema refuses anything else
with raises(TypeError, match="property 'value' does not hold a value of Expressions.Framework.Term"):
    term(lambda u: u.literal(lambda r: r.value(lit)))
with raises(TypeError, match="property 'value' does not hold a value of a container"):
    term(lambda u: u.literal(lambda r: r.value(op.call.arguments)))

## EXP-02 · Calls have a callable and ordered arguments

In [ ]:
form = F.form(op)
assert form.kind == "call" and form.attributes == {"callable": "f"} and [a.literal.value.get() for a in form.arguments] == [1, "x", 3]
assert F.same(form.arguments[2], lit) and E.DIALECT.validate(op) == [] and op.call.callable.get() == "f"
nested = term(lambda u: u.call(lambda c: c.callable("not").arguments([E.DIALECT.spec(op)])))
assert F.same(F.form(nested).arguments[0], op)
assert E.DIALECT.validate(term(lambda u: u.call(lambda c: c))) == ["a call needs a callable"]
assert E.DIALECT.validate(term(lambda u: u.call(lambda c: c.callable("")))) == ["a call needs a callable"]
assert E.DIALECT.validate(term(lambda u: u.call(lambda c: c.callable(3)))) == ["a call's callable must be a str, got int"]
bad = term(E.operation("f", E.literal(None), lambda u: u.call(lambda c: c.arguments([E.literal(None)]))))
assert E.DIALECT.validate(bad) == [
    "argument 0: a literal needs a value",
    "argument 1: a call needs a callable",
    "argument 1: argument 0: a literal needs a value",
]

## EXP-03 · A term is a nested value: written into another it is copied, so it holds no cycle

In [ ]:
pair = term(E.operation("g", op, op))  # each argument a copy of op
assert E.DIALECT.validate(pair) == [] and all(F.same(a, op) for a in F.form(pair).arguments)
assert len(list(F.walk(pair))) == 1 + 2 * len(list(F.walk(op)))  # nothing is shared: every term is visited
again = term(pair)
assert F.same(again, pair) and again is not pair

## EXP-04 · Terms are built through mbse-schemas' builders, in a transaction: a term alone, or an expression's root

In [ ]:
x = E.variable("x")
with store.transaction():
    source = store.Expressions.Framework.Expression(lambda e: e.root(E.operation("f", 1)))
assert Expression.ref and F.form(F.term(source)).attributes == {"callable": "f"}
with store.transaction():
    updated = store.Expressions.Framework.Expression(source, lambda e: e.root(E.operation("g", 2)))  # an update
    copy = store.Expressions.Framework.Expression(source)  # a clone
assert updated is source and F.form(F.term(source)).attributes == {"callable": "g"}
assert copy is not source and F.same(copy, source)
with raises(ValueError, match="no term is set"):
    with store.transaction():
        F.term(store.Expressions.Framework.Expression(lambda e: e))
assert F.same(F.term(source.root), source)  # an expression, its root property, or the term itself
with raises(TypeError, match="not a term: 3"):
    F.term(3)
# A spec is a writer, a term value (copied), a native value (a literal) or a callable on the term value's builder.
assert isinstance(E.DIALECT.spec(x), E.Writer) and isinstance(E.DIALECT.spec(lit), F.Writer)
assert isinstance(E.DIALECT.spec(1), E.Writer) and isinstance(E.DIALECT.spec(lambda u: u.variable(lambda v: v.name("x"))), F.Writer)
with raises(TypeError, match="expected a term, a writer or a native value, got <class 'int'>"):
    E.DIALECT.spec(int)
with raises(TypeError, match="expected a term, a writer or a native value, got [1]"):
    E.DIALECT.spec([1])
with raises(TypeError, match="expected a term, a writer or a native value, got None"):
    F.spec(None)
with raises(TypeError, match="expected a term, a writer or a native value, got 1"):
    F.spec(1, lambda value: None)  # a dialect with no literal for it

## EXP-05 · The framework's schemas are an install: ordinary, registered schemas, which Basic's install extends

In [ ]:
before = JSON.ToJSON(store).Snapshot()
store.transaction(Framework.install)
store.transaction(Basic.install)
assert JSON.ToJSON(store).Snapshot() == before  # run again, each install writes nothing
names = ["Expressions.Framework.Literal", "Expressions.Framework.Call", "Expressions.Framework.Variable",
         "Expressions.Framework.Let", "Expressions.Basic.Quantifier"]
kinds = [store.registered(name) for name in names]
for schema in [Term, Expression, store.registered("Expressions.Framework.HasRoot"), *kinds]:
    assert schema.validate() == []
assert [b.name for b in Term.branches] == ["literal", "call", "variable", "let", "quantifier"]
assert [b.schema for b in Term.branches] == kinds and not any(kind.ref for kind in kinds)  # value objects
assert [list(k.properties) for k in kinds] == [["value", "domain"], ["callable", "arguments", "domain"], ["name", "domain"],
                                               ["name", "value", "body", "domain"],
                                               ["name", "quantifier", "collection", "body", "domain"]]
assert kinds[1].properties["arguments"].schema.element is Term and kinds[3].properties["body"].schema is Term
assert Expression.ref and list(Expression.properties) == ["root"] and Expression.properties["root"].schema is Term
HasRoot = store.registered("Expressions.Framework.HasRoot")
assert list(HasRoot.entry.links) == ["schema", "expression"] and HasRoot.uniques == (frozenset({"expression"}),)
OfExpression = store.registered("Schemas.OfExpression")
assert OfExpression.adjacencies["expression"].relation is HasRoot and OfExpression.adjacencies["expression"].me == "schema"

## EXP-06 · Round trips through Plain, JSON and YAML: a term written nested, its kind its branch's name

In [ ]:
expr = term(E.operation("f", E.operation("eq", 1, 1.5), "s", b"\x01", True))
expected = JSON.ToJSON(store)(Term, expr)
for encode, decode in [(Plain.ToPlain(store), Plain.FromPlain(store)), (JSON.ToJSON(store), JSON.FromJSON(store)),
                       (YAML.ToYAML(store), YAML.FromYAML(store))]:
    back = decode(Term, encode(Term, expr))
    assert back is not expr and F.same(back, expr) and E.DIALECT.validate(back) == []
    assert JSON.ToJSON(store)(Term, back) == expected
assert JSON.ToJSON(store)(Term, term(E.operation("not", True))) == (
    '{"call": {"callable": "not", "arguments": [{"literal": {"value": true}}]}}')
alone = JSON.ToJSON(store)(Term, term(-0.0))
assert alone == '{"literal": {"value": -0.0}}'
assert math.copysign(1.0, JSON.FromJSON(store)(Term, alone).literal.value.get()) == -1.0
# A literal's value is bare where only one basic scalar writes its form, otherwise wrapped, kind then schema.
for value, wire in [(1, '1'), (1.5, '1.5'), (True, 'true'), ("x", '{"scalar": {"str": "x"}}'),
                    (b"\x01", '{"scalar": {"bytes": "AQ=="}}'), (math.nan, '{"scalar": {"float": "NaN"}}')]:
    written = JSON.ToJSON(store)(Term, term(value))
    assert written == '{"literal": {"value": %s}}' % wire and F.same(JSON.FromJSON(store)(Term, written), term(value))
# nothing set: only the branch is written
assert [JSON.ToJSON(store)(Term, term(lambda u, k=k: getattr(u, k)(lambda r: r))) for k in ("literal", "call", "variable", "let")] == [
    '{"literal": {}}', '{"call": {}}', '{"variable": {}}', '{"let": {}}']
with raises(Exception, match="has no branch 'operation'"):
    JSON.FromJSON(store)(Term, '{"operation": {}}')
# A store holding expressions round-trips whole, read into a store with the same installs.
with store.transaction():
    store.Expressions.Framework.Expression(lambda e: e.root(expr))
snapshot = JSON.ToJSON(store).Snapshot()
again = Builders.Store()
again.transaction(Framework.install)
again.transaction(Basic.install)
JSON.FromJSON(again).Snapshot(snapshot)
assert JSON.ToJSON(again).Snapshot() == snapshot

## EXP-07 · An expression is a reference object; a schema's expression is linked by `HasRoot`, an extension adjacency

In [ ]:
with store.transaction():
    adult = store.Expressions.Framework.Expression(lambda e: e.root(E.variable("this").age.ge(18)))
    schema = store.Schemas.OfExpression(lambda s: s.name("exp.Adult").result([store.Schemas.OfBool])
                                        .expression(lambda a: a.insert(lambda e: e.expression(adult))))
linked = [entry.expression for entry in Builders.OfExpression.Builder(schema, False, store).expression]
assert len(linked) == 1 and F.same(linked[0], adult)
assert Validators.Validate(store)(Expression, adult) == []
assert V.OfAny(adult, {"this": None}) is None

## EXP-08 · Validation and comparison work through the schemas

In [ ]:
assert Validators.Validate(store)(Expression, adult) == [] and E.DIALECT.validate(adult, bound={"this"}) == []


def compare(a, b):
    return C.OfUnion(Term, term(a)).compare(C.OfUnion(Term, term(b)))


assert compare(1, 1) == 0 and compare(1, 2) is None  # values are not ordered
assert compare(1, 1.0) is None  # another scalar
assert compare(E.operation("f", 1), E.operation("f", 1)) == 0 and compare(E.operation("f", 1), E.operation("f", 2)) is None
assert compare(E.variable("x"), E.literal("x")) is None  # another branch

## EXP-09 · Terms read and write through the visitor protocols

In [ ]:
names = [p.as_schema().name for p in op.call.as_properties]
assert names == ["callable", "arguments"]  # the properties set, schema unaware
entries = []
op.call.arguments.as_value(lambda a: a.as_container(lambda c: c.as_entries(entries.extend)))
assert [i for i, _ in entries] == [0, 1, 2]
values = []
for _, element in entries:
    element.as_union(lambda u: values.append(u.literal.value.get()))
assert values == [1, "x", 3] and F.same(F.term(entries[0][1]), term(1)) and not F.is_term(entries[0][1])  # an element's reader holds a term
seen = []
term(lambda u: (seen.append(type(u.as_schema()).__name__), u.literal(lambda r: (seen.append(r.as_schema().name), r)[1]))[1])
assert seen == ["_UnionData", "Expressions.Framework.Literal"]
# A writer writes through what it is given: a term value's union, a property, a container's element.
with store.transaction():
    written = store.Expressions.Framework.Expression(lambda e: e.root(lambda p: p.as_value(lambda a: a.as_union(
        lambda u: u.call(lambda c: c.callable("f").arguments([E.literal(1), lambda y: y.as_union(E.variable("x").write)]))))))
assert F.same(written, term(E.operation("f", 1, E.variable("x"))))

## EXP-10 · Variables and lets

In [ ]:
v = term(E.variable("x"))
assert F.name_of(v) == "x" and F.kind_of(v)[0] == "variable"
assert E.DIALECT.validate(v) == ["variable 'x' is not bound"] and E.DIALECT.validate(v, bound={"x"}) == []
assert E.DIALECT.validate(term(lambda u: u.variable(lambda r: r))) == ["a variable needs a name"]
assert E.DIALECT.validate(term(lambda u: u.variable(lambda r: r.name(1)))) == ["a variable's name must be a str, got int"]
let = term(E.let_("x", 1, E.variable("x")))
form = F.form(let)
assert form.attributes == {"name": "x"} and F.same(form.arguments[0], term(1)) and F.same(form.arguments[1], v)
assert E.DIALECT.validate(let) == [] and F.name_of(let) == "x"
assert E.DIALECT.validate(term(E.let_("x", None, None))) == ["a let needs a value", "a let needs a body"]
assert E.DIALECT.validate(term(lambda u: u.let(lambda r: r.value(E.literal(1)).body(E.variable("x"))))) == [
    "a let needs a name", "body: variable 'x' is not bound"]
same_name = term(E.let_("y", E.variable("y"), E.variable("y")))
assert E.DIALECT.validate(same_name) == ["value: variable 'y' is not bound"]  # the name is bound in the body only
open_let = term(E.let_("x", None, None))
assert F.form(open_let).arguments == (None, None) and F.name_of(term(E.literal(1))) is None
assert F.same(term(E.DIALECT.make(F.form(open_let))), open_let)

## EXP-11 · Writers build terms with methods

In [ ]:
this = E.variable("this")
writer = this.age.ge(18)
assert isinstance(writer, E.Writer) and callable(writer)
form = F.form(term(writer))
assert form.attributes == {"callable": "ge"} and form.arguments[1].literal.value.get() == 18
got = F.form(form.arguments[0])
assert got.attributes == {"callable": "get"} and F.name_of(got.arguments[0]) == "this" and got.arguments[1].literal.value.get() == "age"
assert F.form(term(this.get("eq"))).arguments[1].literal.value.get() == "eq"  # a property named like a method
methods = [this.eq(1), this.ne(1), this.lt(1), this.le(1), this.gt(1), this.ge(1), this.and_(True), this.or_(True),
           this.not_(), this.implies(True), this.add(1), this.sub(1), this.mul(1), this.neg(), this.has("x"),
           this.get("x")]
assert [F.form(term(m)).attributes["callable"] for m in methods] == [
    "eq", "ne", "lt", "le", "gt", "ge", "and", "or", "not", "implies", "add", "sub", "mul", "neg", "has", "get"]
assert term(E.literal(3)).literal.value.get() == 3 and F.form(term(E.operation("in", 1, 2))).attributes == {"callable": "in"}
bound = term(E.let_("a", 1, E.variable("a")))
assert F.name_of(bound) == "a" and F.same(term(bound), bound)
with raises(AttributeError):
    this._private  # Python's own probes are not properties

## EXP-12 · `Python.Text.FromFunction` reads an expression from a Python function (Python only)

In [ ]:
import linecache

from mbse.Expressions import Translators as T
from mbse.Expressions.Dialects import Python
from mbse.Expressions.Dialects.Python import Expressions as P, Text as PText

store.transaction(Python.install)  # the translation below writes Python's terms


def same_as(function, writer):
    """Whether the terms a function and a writer write are the same: by their kinds, attributes and arguments."""
    got, expected = term(PText.FromFunction(function)), term(writer)
    assert F.same(got, expected), (JSON.ToJSON(store)(Term, got), JSON.ToJSON(store)(Term, expected))


MIN = 18
this, t, a, one = E.variable("this"), E.variable("t"), E.variable("a"), E.literal(1)
same_as(lambda this: this.age >= MIN and this.email is not None, this.age.ge(18).and_(this.has("email")))
same_as(lambda t: t.a < 1 < t.b, t.a.lt(one).and_(one.lt(t.b)))  # a chain; the middle operand written in each
same_as(lambda t: -t.a * 2 + +t.b - 1 == 0, t.a.neg().mul(2).add(t.b).sub(1).eq(0))
same_as(lambda t: (lambda a: a.b > a.c)(t.d), E.let_("a", t.d, a.b.gt(a.c)))  # a let


def constraint(t):
    """A def works too, if its body is one return."""
    return not hasattr(t, "x") or getattr(t, "y") != 2 or t.z is None


same_as(constraint, t.has("x").not_().or_(t.get("y").ne(2)).or_(t.has("z").not_()))


def plain(t):
    return t.a


same_as(plain, t.a)
first, second = (lambda t: t.a), (lambda t: t.b)  # two on one line
same_as(first, t.a)
same_as(second, t.b)
LIMIT, LIMIT_TERM = E.literal(5), term(E.literal(6))
assert PText.FromFunction(lambda: LIMIT) is LIMIT and isinstance(PText.FromFunction(lambda t: t.a), E.Writer)  # a writer
assert F.same(term(PText.FromFunction(lambda: LIMIT_TERM)), LIMIT_TERM)  # a term value, copied where it is written


def at_least(n):
    return lambda t: t.a >= n  # a closure: n is read when FromFunction runs


same_as(at_least(3), t.a.ge(3))


def beside():
    def helper(v):  # a nested def of several statements, beside the lambda
        w = v
        return w
    return helper(lambda t: t.a > 1)


same_as(beside(), t.a.gt(1))
assert V.OfAny(term(PText.FromFunction(lambda: 1 == 1.0))) is None  # the expression's rules, not Python's
with store.transaction():  # written where a term is written: an expression's root
    adult = store.Expressions.Framework.Expression(lambda e: e.root(PText.FromFunction(lambda this: this.age >= 18)))
assert F.same(adult.root, term(this.age.ge(18)))

# Collections, in the forms the Basic-to-Python translator writes, so its text reads back.
x = E.variable("x")  # each t.xs below is its own term, as in the lambdas
same_as(lambda t: all(x > 0 for x in t.xs), t.xs.all("x", x.gt(0)))
same_as(lambda t: any(x > 0 for x in t.xs if x < 5), t.xs.any("x", x.lt(5).and_(x.gt(0))))
same_as(lambda t: all(x > 0 for x in t.xs if x < 5 if x > 1), t.xs.all("x", x.lt(5).and_(x.gt(1)).implies(x.gt(0))))
same_as(lambda t: sum(1 for x in t.xs if x > 8.0) <= 2, t.xs.count_where("x", x.gt(8.0)).le(2))
same_as(lambda t: sum(1 for x in t.xs), t.xs.count_where("x", True))
same_as(lambda t: len(t.xs) > 0 and t.xs[0] == 1 and 3 in t.xs and 4 not in t.xs,
        t.xs.count().gt(0).and_(t.xs.item(0).eq(1)).and_(E.literal(3).in_(t.xs)).and_(E.literal(4).in_(t.xs).not_()))
same_as(lambda t: sum(t.xs) + max(t.xs) - min(t.xs), t.xs.sum().add(t.xs.max()).sub(t.xs.min()))
same_as(lambda t: all(any(y == x for y in t.ys) for x in t.xs), t.xs.all("x", t.ys.any("y", E.variable("y").eq(x))))
c = E.variable("c")
constraint = t.xs.count_where("c", c.gt(8.0)).le(2).and_(t.xs.max().le(12.0)).and_(t.xs.all("c", c.ge(2.0))).and_(
    t.xs.item(1).eq(1.0)).and_(E.literal(1.0).in_(t.xs)).and_(t.xs.count().gt(0))
written = ("sum(1 for c in t.xs if c > 8.0) <= 2 and max(t.xs) <= 12.0 and all(c >= 2.0 for c in t.xs) and t.xs[1] == 1.0 "
           "and 1.0 in t.xs and len(t.xs) > 0")
assert PText.ToText(term(T.between(E.DIALECT, P.DIALECT).forward(term(constraint)))) == written  # which the function writes
same_as(lambda t: sum(1 for c in t.xs if c > 8.0) <= 2 and max(t.xs) <= 12.0 and all(c >= 2.0 for c in t.xs) and
        t.xs[1] == 1.0 and 1.0 in t.xs and len(t.xs) > 0, constraint)  # each quantifier with its own c


def shadowed(min):
    return lambda t: min(t.xs)  # min is not Python's own here


for function, message in [
    (lambda t: t.a / 2, "cannot convert 't.a / 2': not supported in an expression"),
    (lambda t: t.a if t else 1, "cannot convert 't.a if t else 1': not supported in an expression"),
    (lambda t: ~t.a, "cannot convert '~t.a': not supported in an expression"),
    (lambda t: t.a is not t, "cannot convert 't.a is not t': 'is' only tests whether a property is None"),
    (lambda: None, "cannot convert 'None': only native constants are literals"),
    (lambda: undefined_name, "cannot convert 'undefined_name': the name is not defined"),  # noqa: F821
    (lambda: linecache, "cannot convert 'linecache': a module is not a native value or an expression"),
    (lambda t=1: t, "only plain positional parameters can become variables"),
    (lambda: (lambda a, b: a)(1, 2), "cannot convert '(lambda a, b: a)(1, 2)': a let binds one name"),
    (lambda t: hasattr(t, 1), "cannot convert 'hasattr(t, 1)': hasattr takes an object and a property name"),
    (lambda t: hasattr(t, "x", y=1), """cannot convert "hasattr(t, 'x', y=1)": not supported in an expression"""),
    (lambda t: len(t, 1), "cannot convert 'len(t, 1)': len takes one collection"),
    (lambda t: all(t.xs), "cannot convert 'all(t.xs)': all takes a generator expression"),
    (lambda t: sum((x for x in t.xs)), "cannot convert 'sum((x for x in t.xs))': a quantifier is all"),
    (lambda t: all((x for x, y in t.xs)), "cannot convert 'all((x for x, y in t.xs))': a generator has one for"),
    (lambda t: any((x for x in t.xs for y in x)), "a generator has one for, over a name"),
    (lambda t: t.xs[0:1], "cannot convert 't.xs[0:1]': slices are not supported"),
    (lambda t: 1 < t.a is None, "cannot convert '1 < t.a is None': not supported in an expression"),
    (lambda len: len(1), "cannot convert 'len(1)': not supported in an expression"),
    (shadowed(max), "cannot convert 'min(t.xs)': not supported in an expression"),
    (lambda hasattr: hasattr(1, "x"), """cannot convert "hasattr(1, 'x')": not supported in an expression"""),
]:
    with raises(ValueError, match=message):
        PText.FromFunction(function)


def two_statements(t):
    t = t
    return t


def bare(t):
    return


for function in [two_statements, bare]:
    with raises(ValueError, match=f"the body of {function.__name__!r} must be a single return statement"):
        PText.FromFunction(function)
with raises(TypeError, match="expected a Python function, got builtin_function_or_method"):
    PText.FromFunction(len)
with raises(ValueError, match="the function's source is not available"):
    PText.FromFunction(eval("lambda: 1"))
linecache.cache["<unparsable>"] = (9, None, ["def (\n"], "<unparsable>")
with raises(ValueError, match="the function's source is not available"):
    PText.FromFunction(eval(compile("lambda: 1", "<unparsable>", "eval")))